# 05 SELECCIÓN MODELO

In [2]:
import os
import sys
from hashlib import sha256
from pathlib import Path
from time import perf_counter

import mlflow
import mlflow.sklearn
import pandas as pd

from mlflow import MlflowClient
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.svm import SVC

directorio = Path.cwd().resolve()
raiz = next(
    (
        ruta for ruta in [directorio, *directorio.parents]
        if (ruta / "src" / "pipeline.py").is_file()
    ),
    None,
)

if raiz is None:
    raise FileNotFoundError("Abre 05_seleccion.ipynb desde el repositorio pset3.")

sys.path.insert(0, str(raiz))

from src.datos import cargar_obt
from src.features import construir_features
from src.pipeline import crear_pipeline

SEMILLA = 42
HORIZONTE = 5
CUANTIL_TARGET = 0.80
inicio_prueba = pd.Timestamp("2023-07-11")
capital_inicial = 10_000.0

mlflow.set_tracking_uri("http://127.0.0.1:5000")
experimento = mlflow.set_experiment(
    "Sistema de alerta temprana de volatilidad para acciones"
)
cliente = MlflowClient()

print("Raíz:", raiz)
print("URI de MLflow:", mlflow.get_tracking_uri())
print("Experimento:", experimento.name)

c:\Users\usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\pydantic\_internal\_fields.py:132: UserWarning: Field "model_name" in PromptModelConfig has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


Raíz: C:\USFQ\Maestría\Ciencia_Datos\pset3
URI de MLflow: http://127.0.0.1:5000
Experimento: Sistema de alerta temprana de volatilidad para acciones


In [3]:
df = cargar_obt()
snapshot = df.sort_values(["TRADE_DATE", "SYMBOL"])

datos_corrida = {
    "tabla_origen": "PSET2_DB.OBT.AAPL_DAILY_OBT",
    "fecha_corte_snapshot": os.environ["DATA_CUTOFF"],
    "sha256_snapshot_obt": sha256(
        snapshot.to_csv(index=False, lineterminator="\n").encode("utf-8")
    ).hexdigest(),
    "filas_snapshot": len(snapshot),
}

print("Dimensiones de la OBT:", df.shape)
print(datos_corrida)

Dimensiones de la OBT: (4087, 19)
{'tabla_origen': 'PSET2_DB.OBT.AAPL_DAILY_OBT', 'fecha_corte_snapshot': '2026-09-30', 'sha256_snapshot_obt': '9e9d8a678f5feb51385ae330e460a538835644c6e2068e9f989fd362ce45d4e8', 'filas_snapshot': 4087}


In [4]:
features = construir_features(df)

vol_futura_5 = (
    features["RET_LAG_0"]
    .rolling(HORIZONTE, min_periods=HORIZONTE)
    .std(ddof=1)
    .shift(-HORIZONTE)
    .rename("VOL_FUTURA_5")
)

fin_ventana_target = features.index.to_series().shift(-HORIZONTE)

datos_modelado = (
    features.iloc[60:]
    .join(vol_futura_5)
    .dropna(subset=["VOL_FUTURA_5"])
)

entrenamiento = datos_modelado.loc[
    fin_ventana_target.reindex(datos_modelado.index) < inicio_prueba
].copy()

fechas_holdout = datos_modelado.index[datos_modelado.index >= inicio_prueba]

X_desarrollo = entrenamiento.drop(columns="VOL_FUTURA_5")
vol_desarrollo = entrenamiento["VOL_FUTURA_5"]
columnas_numericas = X_desarrollo.columns.drop("MES").tolist()
cv_temporal = TimeSeriesSplit(n_splits=5, gap=HORIZONTE)

hash_desarrollo = sha256(
    entrenamiento.to_csv(index=True, lineterminator="\n").encode("utf-8")
).hexdigest()

# Retorno de t a t+1, expresado como proporción y alineado con t.
retorno_siguiente = features["RET_LAG_0"].shift(-1).div(100)

assert fin_ventana_target.loc[X_desarrollo.index].max() < fechas_holdout.min()

print("Desarrollo:", X_desarrollo.shape)
print("Fechas reservadas para prueba:", len(fechas_holdout))

Desarrollo: (3212, 18)
Fechas reservadas para prueba: 805


In [5]:
ids_tuning = {
    "Random Forest": "8e36ee19a4ac415cb8fd7baef1a64261",
    "Regresión logística": "38caf344f8424927a1ee39f6aabbcdc6",
    "SVM": "5b22d6fe28cc410aa12e4221f623147a",
}

estimadores_default = {
    "Random Forest": RandomForestClassifier(random_state=SEMILLA),
    "Regresión logística": LogisticRegression(random_state=SEMILLA),
    "SVM": SVC(probability=True, random_state=SEMILLA),
}

filas = []

for nombre, run_id in ids_tuning.items():
    run = cliente.get_run(run_id)
    parametros = run.data.params
    metricas = run.data.metrics
    tags = run.data.tags

    if (
        run.info.status != "FINISHED"
        or run.info.experiment_id != experimento.experiment_id
        or tags.get("fase") != "tuning"
        or tags.get("mlflow.parentRunId")
        or not tags.get("mejor_run_hijo")
    ):
        raise ValueError(f"Revisa el Run ID padre de {nombre}: {run_id}")

    if parametros.get("sha256_dataset_desarrollo") != hash_desarrollo:
        raise ValueError(f"El dataset actual difiere del usado para {nombre}.")

    if tags["mejora_segun_pset"] != "false":
        raise ValueError(f"Revisa qué configuración conservar para {nombre}.")

    filas.append({
        "modelo": nombre,
        "version_elegida": "default",
        "run_id_elegido": parametros["run_id_default"],
        "auc_elegido": metricas["roc_auc_default"],
        "std_elegida": metricas["roc_auc_default_std_cv"],
    })

comparacion = (
    pd.DataFrame(filas)
    .sort_values("auc_elegido", ascending=False, kind="stable")
    .reset_index(drop=True)
)
finalistas = comparacion.head(2).copy()

display(comparacion.round(4))
print(f"Diferencia de ROC-AUC: {finalistas['auc_elegido'].diff().abs().iloc[1]:.4f}")

,modelo,version_elegida,run_id_elegido,auc_elegido,std_elegida
0,Random Forest,default,88a42b72b1d74abeb5e513798f264ef0,0.7218,0.0484
1,Regresión logística,default,b1d407af9d61477c90cf10b99db6df1a,0.7099,0.0679
2,SVM,default,86557cb8140b4a7b9af5514ccb3679ad,0.6995,0.0401


Diferencia de ROC-AUC: 0.0119


In [6]:
predicciones_cv = {}

for nombre in finalistas["modelo"]:
    bloques = []

    for pliegue, (idx_train, idx_val) in enumerate(
        cv_temporal.split(X_desarrollo), start=1
    ):
        X_train = X_desarrollo.iloc[idx_train]
        X_val = X_desarrollo.iloc[idx_val]
        vol_train = vol_desarrollo.iloc[idx_train]
        vol_val = vol_desarrollo.iloc[idx_val]

        p80 = vol_train.quantile(CUANTIL_TARGET)
        y_train = vol_train.gt(p80).astype(int)
        y_val = vol_val.gt(p80).astype(int)

        pipeline = crear_pipeline(
            clone(estimadores_default[nombre]),
            columnas_numericas,
        )
        pipeline.fit(X_train, y_train)
        proba_alta = pipeline.predict_proba(X_val)[:, 1]

        bloques.append(pd.DataFrame({
            "pliegue": pliegue,
            "clase_alta": y_val,
            "proba_alta": proba_alta,
            "retorno_siguiente": retorno_siguiente.loc[X_val.index],
        }, index=X_val.index))

    predicciones_cv[nombre] = pd.concat(bloques).sort_index()
    print(f"{nombre}: {len(predicciones_cv[nombre]):,} predicciones de validación")

Random Forest: 2,675 predicciones de validación
Regresión logística: 2,675 predicciones de validación


In [7]:
import numpy as np
from sklearn.metrics import recall_score

EXPOSICION_CON_ALERTA = 0.75
COSTO_TRANSACCION = 0.0
umbrales = [*np.linspace(0.05, 0.95, 19), 1.01]


def simular_estrategia(retornos, alertas):
    alerta_vigente = alertas.rolling(HORIZONTE, min_periods=1).max().astype(bool)
    exposicion = pd.Series(1.0, index=retornos.index).mask(
        alerta_vigente, EXPOSICION_CON_ALERTA
    )

    exposicion_previa = exposicion.shift(1, fill_value=1.0)
    costo = exposicion.sub(exposicion_previa).abs() * COSTO_TRANSACCION
    retorno_neto = exposicion.mul(retornos).sub(costo)

    return capital_inicial * retorno_neto.add(1).cumprod()


resultados_umbrales = []

for nombre, predicciones in predicciones_cv.items():
    for umbral in umbrales:
        diferencias = []
        recalls = []

        for _, bloque in predicciones.groupby("pliegue", sort=True):
            retornos = bloque["retorno_siguiente"]
            alertas = bloque["proba_alta"].ge(umbral)

            capital_alertas = simular_estrategia(retornos, alertas)
            capital_buy_hold = capital_inicial * retornos.add(1).cumprod()

            diferencias.append(
                capital_alertas.iloc[-1] - capital_buy_hold.iloc[-1]
            )
            recalls.append(
                recall_score(bloque["clase_alta"], alertas, zero_division=0)
            )

        resultados_umbrales.append({
            "modelo": nombre,
            "umbral_alerta": umbral,
            "diferencia_capital_cv_usd": np.mean(diferencias),
            "recall_cv": np.mean(recalls),
        })

tabla_umbrales = pd.DataFrame(resultados_umbrales)

umbrales_elegidos = (
    tabla_umbrales.sort_values(
        ["diferencia_capital_cv_usd", "recall_cv", "umbral_alerta"],
        ascending=[False, False, False],
    )
    .drop_duplicates("modelo")
    .reset_index(drop=True)
)

display(umbrales_elegidos.round(4))

,modelo,umbral_alerta,diferencia_capital_cv_usd,recall_cv
0,Random Forest,1.01,0.0,0.0
1,Regresión logística,1.01,0.0,0.0


In [8]:
filas_activas = []

for fila in tabla_umbrales.itertuples(index=False):
    n_alertas = int(
        predicciones_cv[fila.modelo]["proba_alta"].ge(fila.umbral_alerta).sum()
    )

    if n_alertas > 0:
        filas_activas.append({
            "modelo": fila.modelo,
            "umbral_alerta": fila.umbral_alerta,
            "alertas_cv": n_alertas,
            "diferencia_capital_cv_usd": fila.diferencia_capital_cv_usd,
            "recall_cv": fila.recall_cv,
        })

mejores_con_alertas = (
    pd.DataFrame(filas_activas)
    .sort_values(
        ["diferencia_capital_cv_usd", "recall_cv", "umbral_alerta"],
        ascending=[False, False, False],
    )
    .drop_duplicates("modelo")
)

display(mejores_con_alertas.round(4))

,modelo,umbral_alerta,alertas_cv,diferencia_capital_cv_usd,recall_cv
17,Random Forest,0.90,1,-81.3022,0.0019
36,Regresión logística,0.95,3,-154.7940,0.0018


In [10]:
assert finalistas["version_elegida"].eq("default").all()

X_test = datos_modelado.loc[fechas_holdout, X_desarrollo.columns]
assert X_test.shape == (805, 18)

p80_final = float(vol_desarrollo.quantile(CUANTIL_TARGET))
y_desarrollo_final = vol_desarrollo.gt(p80_final).astype(int)
y_test = datos_modelado.loc[fechas_holdout, "VOL_FUTURA_5"].gt(p80_final).astype(int)

retornos_test = retorno_siguiente.loc[fechas_holdout]
assert retornos_test.notna().all() and y_test.nunique() == 2

umbrales_por_modelo = umbrales_elegidos.set_index("modelo")["umbral_alerta"]
capital_buy_hold = capital_inicial * retornos_test.add(1).cumprod()


def max_drawdown_pct(capital):
    maximos = capital.cummax().clip(lower=capital_inicial)
    return float(-100 * capital.div(maximos).sub(1).min())


modelos_finales = {}
curvas_capital = {"Buy & Hold": capital_buy_hold}
resultados_finales = []

for fila in finalistas.itertuples(index=False):
    nombre = fila.modelo
    umbral = float(umbrales_por_modelo.loc[nombre])
    pipeline = crear_pipeline(
        clone(estimadores_default[nombre]),
        columnas_numericas,
    )

    inicio = perf_counter()
    pipeline.fit(X_desarrollo, y_desarrollo_final)
    segundos_ajuste = perf_counter() - inicio

    inicio = perf_counter()
    proba_alta = pd.Series(
        pipeline.predict_proba(X_test)[:, 1],
        index=fechas_holdout,
    )
    segundos_inferencia = perf_counter() - inicio

    alertas = proba_alta.ge(umbral)
    capital_modelo = simular_estrategia(retornos_test, alertas)

    modelos_finales[nombre] = pipeline
    curvas_capital[nombre] = capital_modelo

    resultados_finales.append({
        "modelo": nombre,
        "run_id_candidato": fila.run_id_elegido,
        "roc_auc_cv": fila.auc_elegido,
        "std_cv": fila.std_elegida,
        "roc_auc_test": roc_auc_score(y_test, proba_alta),
        "umbral_alerta": umbral,
        "alertas_test": int(alertas.sum()),
        "recall_alertas_test": recall_score(y_test, alertas, zero_division=0),
        "capital_final_usd": float(capital_modelo.iloc[-1]),
        "diferencia_vs_buy_hold_usd": float(
            capital_modelo.iloc[-1] - capital_buy_hold.iloc[-1]
        ),
        "max_drawdown_pct": max_drawdown_pct(capital_modelo),
        "segundos_ajuste": segundos_ajuste,
        "ms_inferencia_por_fila": 1000 * segundos_inferencia / len(X_test),
    })

tabla_final = pd.DataFrame(resultados_finales)
display(tabla_final.round(4))

print(f"P80 final: {p80_final:.4f} %")
print(f"Capital final de Buy & Hold: USD {capital_buy_hold.iloc[-1]:,.2f}")
print(f"Drawdown máximo de Buy & Hold: {max_drawdown_pct(capital_buy_hold):.2f} %")

,modelo,run_id_candidato,roc_auc_cv,std_cv,roc_auc_test,umbral_alerta,alertas_test,recall_alertas_test,capital_final_usd,diferencia_vs_buy_hold_usd,max_drawdown_pct,segundos_ajuste,ms_inferencia_por_fila
0,Random Forest,88a42b72b1d74abeb5e513798f264ef0,0.7218,0.0484,0.6348,1.01,0,0.0,18126.4962,0.0,33.3605,7.0202,0.1529
1,Regresión logística,b1d407af9d61477c90cf10b99db6df1a,0.7099,0.0679,0.6439,1.01,0,0.0,18126.4962,0.0,33.3605,0.1898,0.0702


P80 final: 2.1072 %
Capital final de Buy & Hold: USD 18,126.50
Drawdown máximo de Buy & Hold: 33.36 %
